# Quantum Fourier Transform et Quantum Phase Estimation

$$\text{QFT}|j\rangle = \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} e^{2\pi i j k / N} |k\rangle$$

L'algorithme QPE estime la phase $\varphi$ telle que $U|\psi\rangle = e^{2\pi i \varphi}|\psi\rangle$.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import qutip as qt
import matplotlib.pyplot as plt

### QFT récursive

La QFT se décompose récursivement:

$$\text{QFT}_n = (I \otimes \text{QFT}_{n-1})\, \text{CR}_n\, (H \otimes I)$$

où $\text{CR}_n$ est une cascade de portes $C R_k$ avec $R_k = \text{diag}(1, e^{2\pi i / 2^k})$.

In [ ]:
def qft_circuit(n, inverse=False):
    qc = QuantumCircuit(n)
    for i in range(n):
        qc.h(i)
        for j in range(i + 1, n):
            qc.cp(np.pi / 2**(j - i), j, i)
    if inverse:
        qc = qc.inverse()
    return qc

def iqft_circuit(n):
    return qft_circuit(n, inverse=True)

In [ ]:
# Test QFT sur |001>
n = 3
qc_test = QuantumCircuit(n)
qc_test.x(0)
qc_test.append(qft_circuit(n), range(n))
qc_test.measure_all()

backend = AerSimulator()
qc_t = transpile(qc_test, backend)
counts = backend.run(qc_t, shots=2048).result().get_counts()
plot_histogram(counts, title='QFT|001⟩')

### Quantum Phase Estimation

$$|0\rangle^{\otimes n}|\psi\rangle \xrightarrow{\text{QPE}} |\tilde{\varphi}\rangle|\psi\rangle$$

où $\tilde{\varphi}$ est l'estimation binaire de $\varphi$ sur $n$ qubits.

Précision: $|\varphi - \tilde{\varphi}| \leq 2^{-n}$.

In [ ]:
def qpe_circuit(n_phase, unitary, unitary_name='U'):
    n_target = unitary.num_qubits
    q_phase = QuantumRegister(n_phase, 'phase')
    q_target = QuantumRegister(n_target, 'target')
    c = ClassicalRegister(n_phase, 'c')
    qc = QuantumCircuit(q_phase, q_target, c)

    for i in range(n_target):
        qc.h(q_target[i])
    qc.h(q_phase)

    for i in range(n_phase):
        u_pow = unitary.power(2**i)
        u_pow.name = f'{unitary_name}^{2**i}'
        qc.append(u_pow.to_gate().control(), [q_phase[i]] + q_target[:])

    qc.append(iqft_circuit(n_phase), q_phase)
    qc.measure(q_phase, c)
    return qc

In [ ]:
# Test QPE avec U = Z (phase=0.5)
u_gate = QuantumCircuit(1, name='Z')
u_gate.z(0)
unitary = u_gate.to_gate()

n_phase = 4
qc_qpe = qpe_circuit(n_phase, unitary, 'Z')

qc_t = transpile(qc_qpe, backend)
counts = backend.run(qc_t, shots=2048).result().get_counts()
plot_histogram(counts, title='QPE: φ=0.5 avec U=Z')

In [ ]:
# Analyse de précision
def estimate_phase(n_phase, phase_true):
    u = QuantumCircuit(1, name='U')
    u.p(2 * np.pi * phase_true, 0)
    qc = qpe_circuit(n_phase, u.to_gate(), 'U')
    qc_t = transpile(qc, backend)
    counts = backend.run(qc_t, shots=4096).result().get_counts()
    max_bitstr = max(counts, key=counts.get)
    phase_est = int(max_bitstr, 2) / 2**n_phase
    return phase_est, abs(phase_est - phase_true)

phi = 0.25
for n in range(2, 7):
    est, err = estimate_phase(n, phi)
    print(f'n={n}: φ_est={est:.4f}, erreur={err:.2e} (borne 2^{-n}={2**(-n):.2e})')

### Hamiltonien derrière QPE (QuTiP)

L'opérateur d'évolution $U = e^{-iH\tau}$ relie QPE à l'estimation d'énergie.

Soit $H|\psi\rangle = E|\psi\rangle$, alors $U|\psi\rangle = e^{-iE\tau}|\psi\rangle$ et $\varphi = -E\tau / 2\pi$.

$$H = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} = \sigma_x$$

In [ ]:
H = qt.sigmax()
evals, estates = H.eigenstates()
print('Valeurs propres de σ_x:', evals)
print('États propres:')
for i, s in enumerate(estates):
    print(f'  E={evals[i]:.2f}: {s}')

tau = np.pi / 4
U = (-1j * H * tau).expm()
print(f'\nU = exp(-i H τ) avec τ={tau}')
print(U)

In [ ]:
# Vérification: U|ψ⟩ = exp(-i E τ) |ψ⟩
for i, s in enumerate(estates):
    phase = (-evals[i] * tau) / (2 * np.pi)
    phase_norm = phase - np.floor(phase)
    print(f'État |ψ_{i}⟩: E={evals[i]:.2f}, φ={phase_norm:.4f}')

## Questions

**Q1.** Implémenter QPE pour estimer la phase de $U = \text{diag}(1, e^{2\pi i / 3})$. Comparer la précision obtenue pour $n=3,4,5,6$ qubits de phase avec la borne théorique $2^{-n}$.

**Q2.** En utilisant QuTiP, construire $H = \epsilon \sigma_z + \Delta \sigma_x$ (Hamiltonien d'un qubit en champ transverse). Calculer $U = e^{-iH\tau}$, puis utiliser QPE pour retrouver $E_0$ et $E_1$ à différentes précisions.